# Cross-Sectional Momentum on S&P 500

**Research question:** Do past winners continue to outperform past losers in the S&P 500?

**Hypothesis:** Stocks with the highest returns over the past 12 months will outperform stocks with the lowest returns over the following month.

In [16]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

## 1. Data Collection
Download daily adjusted close prices for all S&P 500 constituents.

In [11]:
response = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
data_frame = pd.read_csv(pd.io.common.StringIO(response.text))

list_sp500 = data_frame["Symbol"].tolist()
print(list_sp500[:10])

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## 2. Download Historical Prices
We download daily adjusted close prices for all S&P 500 constituents since 2010. Adjusted close accounts for dividends and stock splits, ensuring consistent return calculations.

In [23]:
# Fix Yahoo Finance ticker format
list_sp500 = [t.replace('.', '-') for t in list_sp500]

data_candles = yf.download(list_sp500, "2010-01-01", auto_adjust=True)

print(data_candles.shape)
print(data_candles.head())

[*********************100%***********************]  503 of 503 completed


(4233, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
Date                                                                        
2010-01-04  19.743505  6.400959  NaN  NaN  17.968187  7.601905  30.610775   
2010-01-05  19.529049  6.412026  NaN  NaN  17.823019  7.576549  30.799948   
2010-01-06  19.459667  6.310036  NaN  NaN  17.922001  7.543795  31.127378   
2010-01-07  19.434441  6.298368  NaN  NaN  18.070475  7.499420  31.098276   
2010-01-08  19.428118  6.340242  NaN  NaN  18.162848  7.484628  30.974573   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
Date                                         ...                              
2010-01-04  37.090000  21.703720  20.029280  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669453  20.137472  ...  1724

In [26]:
price = data_candles["Close"]
print(price.head())

Ticker              A      AAPL  ABBV  ABNB        ABT      ACGL        ACN  \
Date                                                                          
2010-01-04  19.743505  6.400959   NaN   NaN  17.968187  7.601905  30.610775   
2010-01-05  19.529049  6.412026   NaN   NaN  17.823019  7.576549  30.799948   
2010-01-06  19.459667  6.310036   NaN   NaN  17.922001  7.543795  31.127378   
2010-01-07  19.434441  6.298368   NaN   NaN  18.070475  7.499420  31.098276   
2010-01-08  19.428118  6.340242   NaN   NaN  18.162848  7.484628  30.974573   

Ticker           ADBE        ADI        ADM  ...        WY       WYNN  \
Date                                         ...                        
2010-01-04  37.090000  21.703720  20.029280  ...  9.157347  40.493729   
2010-01-05  37.700001  21.669453  20.137472  ...  9.351746  42.956512   
2010-01-06  37.619999  21.628332  20.086567  ...  9.248346  42.393066   
2010-01-07  36.889999  21.457010  19.876534  ...  9.206984  43.298409   
2010-01-

## 3. Compute Monthly Returns
We resample daily prices to monthly frequency and compute log returns. Log returns are additive over time, which simplifies the calculation of cumulative multi-month returns needed for the momentum signal.

In [30]:
monthly_price = price.resample('ME').last()
monthly_log = np.log(monthly_price)
monthly_yield = monthly_log.diff()
print(monthly_yield.head())

Ticker             A      AAPL  ABBV  ABNB       ABT      ACGL       ACN  \
Date                                                                       
2010-01-31       NaN       NaN   NaN   NaN       NaN       NaN       NaN   
2010-02-28  0.115441  0.063347   NaN   NaN  0.024997  0.033538 -0.025199   
2010-03-31  0.089049  0.138431   NaN   NaN -0.029920  0.030223  0.048349   
2010-04-30  0.052949  0.105280   NaN   NaN -0.020889 -0.008826  0.048235   
2010-05-31 -0.113792 -0.016256   NaN   NaN -0.072966 -0.027634 -0.151100   

Ticker          ADBE       ADI       ADM  ...        WY      WYNN       XEL  \
Date                                      ...                                 
2010-01-31       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2010-02-28  0.070231  0.081184 -0.015513  ...  0.012453  0.026945  0.001443   
2010-03-31  0.020566 -0.007693 -0.015791  ...  0.113815  0.176352  0.030028   
2010-04-30 -0.051338  0.037792 -0.033782  ...  0.090731  0.151567  0.025

## 4. Momentum Signal
For each stock, we compute the cumulative log return over the past 12 months. This is our momentum score: stocks with high past returns are "winners", stocks with low past returns are "losers".

In [33]:
annual_yield = monthly_yield.rolling(12).sum()
print(annual_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-06-30  0.128397  0.347792  0.336357  0.078169 -0.384508  0.063920   
2026-07-31  0.194645  0.401541  0.313912  0.134814 -0.154333  0.155411   
2026-08-31  0.208504  0.314775  0.228358  0.339085 -0.161191  0.069015   
2026-09-30  0.300521  0.272078  0.152577  0.279972 -0.281140  0.017806   
2026-10-31  0.185450  0.246367  0.331414  0.261683 -0.127337  0.208884   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-06-30 -0.862819 -0.635007  0.525344  0.400865  ... -0.035783  0.045022   
2026-07-31 -0.455720 -0.356567  0.505358  0.411562  ...  0.033970 -0.084215   
2026-08-31 -0.294409 -0.197439  0.378678  0.289418  ... -0.074988 -0.318965   
2026-09-30 -0.275865 -0.385371  0.491524  0.313044  ... -0.227574 -0.469941   
2026-10

## 5. Portfolio Construction
Each month, we rank all stocks by their 12-month momentum score and sort them into deciles (10 groups). Decile 1 contains the biggest losers, decile 10 the biggest winners.

In [36]:
ranked_yield = annual_yield.rank(axis=1)
print(ranked_yield.tail())

Ticker          A   AAPL   ABBV   ABNB   ABT   ACGL    ACN  ADBE    ADI  \
Date                                                                      
2026-06-30  253.0  382.0  376.0  221.0  45.0  212.0    8.0  19.0  446.0   
2026-07-31  297.0  413.0  366.0  254.0  85.0  273.0   21.0  36.0  442.0   
2026-08-31  326.0  388.0  342.0  405.0  77.0  226.0   38.0  66.0  421.0   
2026-09-30  401.0  384.0  325.0  392.0  68.0  228.0   70.0  40.0  455.0   
2026-10-31  297.0  345.0  402.0  356.0  76.0  315.0  193.0  38.0  478.0   

Ticker        ADM  ...     WY   WYNN    XEL    XOM    XYL    XYZ    YUM  \
Date               ...                                                    
2026-06-30  409.0  ...  143.0  200.0  302.0  341.0  123.0  240.0  232.0   
2026-07-31  418.0  ...  180.0  108.0  229.0  396.0   74.0  191.0  216.0   
2026-08-31  380.0  ...  113.0   28.0  232.0  418.0   55.0  193.0  220.0   
2026-09-30  409.0  ...   87.0   29.0  136.0  437.0   46.0  227.0  151.0   
2026-10-31  359.0  ...  